In [14]:
import sys
import os

# Adjust this path if needed
sys.path.append("..")

In [15]:
from core.tfidf_algorithm import calculate_idf
from core.tfidf_algorithm import create_sparse_vector
from core.tfidf_algorithm import calculate_cosine_similarity

In [16]:
def get_top_k(user_ingredients, k=5):
    
    user_vec = create_sparse_vector(user_ingredients, IDF_WEIGHTS, VOCAB_MAP)
    
    scores = []
    
    for idx, recipe_vec in enumerate(RECIPE_VECTORS):
        score = calculate_cosine_similarity(user_vec, recipe_vec)
        scores.append((idx, score))
    
    # Sort by similarity descending
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    
    top_k = scores[:k]
    
    return top_k

In [2]:
def is_relevant(user_ing, recipe_ing, threshold=0.7):
    
    user_set = set(user_ing)
    recipe_set = set(recipe_ing)
    
    if len(recipe_set) == 0:
        return False
    
    overlap_ratio = len(user_set & recipe_set) / len(recipe_set)
    
    return overlap_ratio >= threshold

In [3]:
def compute_precision_at_k(test_queries, k=5):
    
    total_precision = 0
    
    for user_ing in test_queries:
        
        top_k = get_top_k(user_ing, k)
        
        relevant_count = 0
        
        for idx, _ in top_k:
            recipe_ing = df.iloc[idx]["core_ingredients"]
            
            if is_relevant(user_ing, recipe_ing):
                relevant_count += 1
        
        precision = relevant_count / k
        total_precision += precision
    
    return total_precision / len(test_queries)

In [4]:
def compute_mrr(test_queries):
    
    total_mrr = 0
    
    for user_ing in test_queries:
        
        user_vec = create_sparse_vector(user_ing, IDF_WEIGHTS, VOCAB_MAP)
        
        scores = []
        
        for idx, recipe_vec in enumerate(RECIPE_VECTORS):
            score = calculate_cosine_similarity(user_vec, recipe_vec)
            scores.append((idx, score))
        
        scores = sorted(scores, key=lambda x: x[1], reverse=True)
        
        reciprocal_rank = 0
        
        for rank, (idx, _) in enumerate(scores, start=1):
            
            recipe_ing = df.iloc[idx]["core_ingredients"]
            
            if is_relevant(user_ing, recipe_ing):
                reciprocal_rank = 1 / rank
                break
        
        total_mrr += reciprocal_rank
    
    return total_mrr / len(test_queries)

In [7]:
import pandas as pd
import ast

df = pd.read_csv("E:/6thsem/HealthyBites/data/processed/healthybites_master_dataset_split.csv")

def safe_eval(x):
    try:
        return ast.literal_eval(x)
    except:
        return []

df["core_ingredients"] = df["core_ingredients"].apply(safe_eval)

print("Dataset loaded:", len(df))

Dataset loaded: 200000


In [8]:
import random

TEST_SIZE = 100

test_queries = []

for _ in range(TEST_SIZE):
    rand_idx = random.randint(0, len(df)-1)
    test_queries.append(df.iloc[rand_idx]["core_ingredients"])

In [20]:
ALL_CORES = df["core_ingredients"].tolist()

IDF_WEIGHTS, VOCAB_MAP = calculate_idf(ALL_CORES)

RECIPE_VECTORS = [
    create_sparse_vector(r, IDF_WEIGHTS, VOCAB_MAP)
    for r in ALL_CORES
]

print("TF-IDF vectors created:", len(RECIPE_VECTORS))

TF-IDF vectors created: 200000


In [21]:
precision_5 = compute_precision_at_k(test_queries, k=5)
mrr_score = compute_mrr(test_queries)

print("Precision@5:", round(precision_5, 4))
print("MRR:", round(mrr_score, 4))

Precision@5: 0.428
MRR: 1.0
